# 第 17 章互动实验：合成相机标定

用已知真值的合成相机检验你的标定实现，观察噪声、视图数和覆盖范围对精度的影响。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "17_相机标定"
ASSIGNMENT = "17-1_张正友标定.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def rodrigues(r):
    t = np.linalg.norm(r)
    if t < 1e-12: return np.eye(3)
    k = r / t; K_ = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(t) * K_ + (1 - np.cos(t)) * K_ @ K_

K_TRUE = np.array([[820.0, 0, 318.0], [0, 790.0, 247.0], [0, 0, 1]])
xs, ys = np.meshgrid(np.arange(9) * 15.0, np.arange(6) * 15.0)
BOARD = np.stack([xs.ravel(), ys.ravel()], 1)
BOARD3 = np.hstack([BOARD, np.zeros((len(BOARD), 1))])

def project(K, R, t, P):
    pc = P @ R.T + t; uv = pc @ K.T; return uv[:, :2] / uv[:, 2:3]

def make_views(n, noise, seed, tilt=0.35, spread=60):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        R = rodrigues(rng.uniform(-tilt, tilt, 3))
        t = np.array([rng.uniform(-spread, 0) - 30, rng.uniform(-spread, 0) - 20, rng.uniform(380, 500)])
        out.append((project(K_TRUE, R, t, BOARD3) + rng.normal(scale=noise, size=(len(BOARD), 2)), R, t))
    return out

def run_or_hint(experiment):
    """运行一个实验函数；它用到的作业函数还没完成时，打印提示并返回 None。"""
    try:
        return experiment()
    except NotImplementedError as exc:
        print(f"先完成作业：{exc}")
        return None

views = make_views(6, 0.3, 0)
for uv, _, _ in views:
    plt.plot(uv[:, 0], uv[:, 1], ".", ms=3)
plt.xlim(0, 640); plt.ylim(480, 0); plt.gca().set_aspect("equal"); plt.title("6 个视图的角点"); plt.show()

In [ ]:
def calibrate(views):
    Hs = [mod.homography_dlt(BOARD, uv) for uv, _, _ in views]
    return mod.intrinsics_from_homographies(Hs)

def try_calibrate(views):
    """和 calibrate 一样，但闭式解解不出有效内参时返回 None。
    视图少、噪声大或者姿态太单一时会这样：公式解法在开方时出现 nan，Cholesky 解法报“不是正定矩阵”。"""
    try:
        with np.errstate(all="ignore"):
            K = calibrate(views)
    except np.linalg.LinAlgError:
        return None
    return K if np.all(np.isfinite(K)) else None

K = attempt(calibrate, views)
if K is not None:
    np.set_printoptions(precision=2, suppress=True)
    print("估计 K：\n", K)
    print("真值 K：\n", K_TRUE)

## 实验 1：噪声与视图数

预测：视图从 3 张增加到 15 张，fx 误差大致怎么变？

In [ ]:
def views_and_noise(noises=(0.1, 0.3, 1.0), counts=(3, 5, 10, 15), repeats=20):
    res, failed = {}, []
    for noise in noises:
        for n in counts:
            errs = []
            for s in range(repeats):
                K = try_calibrate(make_views(n, noise, s))
                if K is None:
                    failed.append((noise, n))   # 解不出有效的内参，不计入中位数
                else:
                    errs.append(abs(K[0, 0] - 820) / 820)
            res[(noise, n)] = np.median(errs) if errs else np.nan
    return res, failed

result = run_or_hint(views_and_noise)
if result:
    res, failed = result
    for noise in (0.1, 0.3, 1.0):
        plt.plot((3, 5, 10, 15), [100 * res[(noise, n)] for n in (3, 5, 10, 15)], "o-", label=f"噪声 {noise} px")
    plt.xlabel("视图数"); plt.ylabel("fx 相对误差中位数 (%)"); plt.legend(); plt.show()
    if failed:
        print(f"有 {len(failed)} 次标定解不出有效的内参，没有计入中位数，出现在（噪声, 视图数）= {sorted(set(failed))}。想一想：为什么偏偏是这种条件？")

## 实验 2：姿态太单一会怎样

把倾斜角限制在 ±0.05 rad（几乎正对相机），观察主点和焦距。

In [ ]:
rows = run_or_hint(lambda: [(tilt, try_calibrate(make_views(10, 0.3, 1, tilt=tilt))) for tilt in (0.35, 0.05)])
for tilt, K in rows or []:
    if K is None:
        print(f"倾斜 ±{tilt} rad：解不出有效的内参")
    else:
        print(f"倾斜 ±{tilt} rad：fx={K[0,0]:.1f} fy={K[1,1]:.1f} cx={K[0,2]:.1f} cy={K[1,2]:.1f}")

## 实验 3：重投影误差小不等于标定准

In [ ]:
views = make_views(10, 0.3, 2)

def per_view_rmse():
    K = calibrate(views)
    rmse = []
    for uv, _, _ in views:
        H = mod.homography_dlt(BOARD, uv)
        R, t = mod.extrinsics_from_homography(K, H)
        rmse.append(mod.reprojection_rmse(K, R, t, BOARD3, uv))
    return rmse

rmse = run_or_hint(per_view_rmse)
if rmse is not None:
    print("各视图重投影 RMSE：", np.round(rmse, 3))
    print("想一想：这些误差是用估计的 K 自洽地算出来的，它能发现哪些错误，发现不了哪些？")
if cv2 is not None:
    obj = [BOARD3.astype(np.float32)] * len(views)
    img = [uv.astype(np.float32) for uv, _, _ in views]
    rms, Kcv, dist, _, _ = cv2.calibrateCamera(obj, img, (640, 480), None, None)
    print("OpenCV RMS", round(rms, 3), "\nOpenCV K\n", Kcv.round(2))

## 实验 4：倾斜角变小时，哪个数字会报警

实验 2 只看了一次标定。这次把倾斜角的上限从 0.5 rad 一路降到 0.05 rad，每个角度重复 20 次，同时记录 fx 的相对误差和重投影 RMS（10 张视图，角点噪声 0.3 px，没有畸变）。

预测：倾斜角变小时，fx 误差和重投影 RMS 各怎么变？如果手里只有标定程序报告的 RMS，你能发现姿态太单一吗？

In [ ]:
def linear_calibration(views):
    """用作业里的线性解标定。返回 (K, 所有视图合起来的重投影 RMS)；闭式解解不出有效内参时返回 None。"""
    K = try_calibrate(views)
    if K is None:
        return None
    sq = []
    with np.errstate(all="ignore"):
        for uv, _, _ in views:
            R, t = mod.extrinsics_from_homography(K, mod.homography_dlt(BOARD, uv))
            sq.append(mod.reprojection_rmse(K, R, t, BOARD3, uv) ** 2)
    rms = float(np.sqrt(np.mean(sq)))
    return (K, rms) if np.isfinite(rms) else None

def tilt_experiment(tilts=(0.5, 0.35, 0.2, 0.1, 0.05), repeats=20):
    rows = []
    for tilt in tilts:
        results = [linear_calibration(make_views(10, 0.3, 300 + s, tilt=tilt)) for s in range(repeats)]
        ok = [r for r in results if r is not None]
        fx_err = [abs(K[0, 0] - 820) / 820 * 100 for K, _ in ok]
        rows.append((tilt, len(results) - len(ok), np.median(fx_err) if ok else np.nan, np.median([r for _, r in ok]) if ok else np.nan))
    return rows

rows = run_or_hint(tilt_experiment)
if rows:
    for tilt, failed, fx_err, rms in rows:
        print(f"倾斜上限 ±{tilt:.2f} rad（约 {np.degrees(tilt):.0f}°）：fx 误差中位数 {fx_err:.2f}%，重投影 RMS 中位数 {rms:.3f} px，解不出有效内参 {failed}/20 次")
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    deg = [np.degrees(r[0]) for r in rows]
    ax[0].semilogy(deg, [r[2] for r in rows], "o-"); ax[0].set_xlabel("倾斜上限（度）"); ax[0].set_ylabel("fx 相对误差中位数 (%)")
    ax[1].plot(deg, [r[3] for r in rows], "s-", color="C3"); ax[1].set_ylim(0, 0.6); ax[1].set_xlabel("倾斜上限（度）"); ax[1].set_ylabel("重投影 RMS 中位数 (px)")
    plt.tight_layout(); plt.show()
    print("想一想：RMS 几乎不随倾斜角变，为什么？对照教程第 3.4 节“为什么板子要倾斜”和第 6.1 节。")

## 实验 5：镜头有畸变，模型里没有

给合成相机加上 $k_1 = -0.1$ 的桶形畸变，它让画面四角的点向内移动将近 10 px。拍 40 张覆盖全画面的视图（角点噪声 0.2 px），用三种办法标定：你的线性解（模型里没有畸变）；OpenCV 的非线性优化，但不建畸变模型；OpenCV 建 $k_1$、$k_2$。没装 OpenCV 时只做第一种。

每种办法都把角点的残差沿半径方向的分量，按“到主点的距离”分组取平均。随机噪声在平均后互相抵消，剩下的就是模型没解释掉的部分。

预测：三种办法的重投影 RMS 各是好几个像素、1 px 上下，还是零点几？不建畸变模型时 fx 偏多少？哪一条残差曲线是平的？

In [ ]:
def project_k1(K, R, t, P, k1):
    pc = P @ R.T + t
    x, y = pc[:, 0] / pc[:, 2], pc[:, 1] / pc[:, 2]
    f = 1 + k1 * (x * x + y * y)
    return np.stack([K[0, 0] * x * f + K[0, 2], K[1, 1] * y * f + K[1, 2]], axis=1)

def views_k1(n, noise, seed, k1=-0.1, spread=0.85, tilt=0.5):
    """带径向畸变的视图。spread 是板中心偏离画面中心的范围（0 到 1）；只保留整块板都在画面内的姿态。"""
    rng = np.random.default_rng(seed); out = []
    center = np.append(BOARD.mean(axis=0), 0.0)
    while len(out) < n:
        R = rodrigues(rng.uniform(-tilt, tilt, 3))
        aim = np.array([318 + spread * rng.uniform(-1, 1) * 320, 247 + spread * rng.uniform(-1, 1) * 240, 1.0])
        t = rng.uniform(380, 520) * np.linalg.inv(K_TRUE) @ aim - R @ center
        uv = project_k1(K_TRUE, R, t, BOARD3, k1)
        if uv[:, 0].min() > 2 and uv[:, 0].max() < 638 and uv[:, 1].min() > 2 and uv[:, 1].max() < 478:
            out.append((uv + rng.normal(scale=noise, size=uv.shape), R, t))
    return out

def residuals(K, views):
    """K 固定，逐视图用作业函数求外参。返回所有角点的位置和残差向量（观测位置减重投影位置）。"""
    pos, res = [], []
    for uv, _, _ in views:
        R, t = mod.extrinsics_from_homography(K, mod.homography_dlt(BOARD, uv))
        pos.append(uv); res.append(uv - project(K, R, t, BOARD3))
    return np.vstack(pos), np.vstack(res)

def rms_of(res):
    return float(np.sqrt(np.mean(np.sum(res ** 2, axis=1))))

EDGES = np.arange(0, 351, 50)

def radial_means(pos, res):
    """残差沿半径方向（从主点指向角点）的分量，按角点到主点的距离每 50 px 一组取平均。"""
    offset = pos - K_TRUE[:2, 2]
    radius = np.hypot(offset[:, 0], offset[:, 1])
    radial = (offset * res).sum(axis=1) / np.maximum(radius, 1e-9)
    return [radial[(radius >= a) & (radius < b)].mean() for a, b in zip(EDGES[:-1], EDGES[1:])]

def cv_calibrate(views, with_distortion):
    """OpenCV 的 calibrateCamera。返回 K、畸变系数、RMS，以及所有角点的位置和残差。"""
    obj = BOARD3.astype(np.float32)
    img = [uv.astype(np.float32).reshape(-1, 1, 2) for uv, _, _ in views]
    flags = cv2.CALIB_ZERO_TANGENT_DIST | cv2.CALIB_FIX_K3
    if not with_distortion:
        flags |= cv2.CALIB_FIX_K1 | cv2.CALIB_FIX_K2
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera([obj] * len(img), img, (640, 480), None, None, flags=flags)
    res = [uv - cv2.projectPoints(obj, rv, tv, K, dist)[0].reshape(-1, 2) for (uv, _, _), rv, tv in zip(views, rvecs, tvecs)]
    return K, dist, float(rms), np.vstack([uv for uv, _, _ in views]), np.vstack(res)

distorted = views_k1(40, 0.2, 3)
curves = {}

def linear_on_distorted():
    K = try_calibrate(distorted)
    if K is None:
        return "解不出有效的内参"
    pos, res = residuals(K, distorted)
    return K, rms_of(res), radial_means(pos, res)

result = run_or_hint(linear_on_distorted)
if isinstance(result, str):
    print("你的线性解：" + result)
elif result:
    K, rms, curves["线性解，没有畸变模型"] = result
    print(f"你的线性解：重投影 RMS {rms:.2f} px，fx={K[0, 0]:.1f}（真值 820，偏 {100 * (K[0, 0] - 820) / 820:+.1f}%）")
if cv2 is not None:
    for name, with_distortion in (("OpenCV 优化，没有畸变模型", False), ("OpenCV 优化，建 k1、k2", True)):
        K, dist, rms, pos, res = cv_calibrate(distorted, with_distortion)
        curves[name] = radial_means(pos, res)
        print(f"{name}：重投影 RMS {rms:.2f} px，fx={K[0, 0]:.1f}（偏 {100 * (K[0, 0] - 820) / 820:+.1f}%），k1={dist.ravel()[0]:+.3f}")
if curves:
    for name, means in curves.items():
        plt.plot(EDGES[:-1] + 25, means, "o-", label=name)
    plt.axhline(0, color="k", lw=0.5); plt.xlabel("角点到主点的距离 (px)"); plt.ylabel("径向残差的平均 (px)"); plt.legend(); plt.show()
    if cv2 is not None:
        print("想一想：优化之后、没有畸变模型的那一次，RMS 看上去很正常，只看 RMS 你会放过它吗？它的 fx 和残差曲线说明了什么？（教程第 6.2 节）")
    else:
        print("想一想：畸变让四角的点移动了将近 10 px，线性解的 RMS 却小得多，差的那部分被谁吸收了？装上 OpenCV 后重跑，可以看到另外两条曲线。（教程第 6.2 节）")

## 实验 6：留出验证，按位置看残差

相机仍带 $k_1 = -0.1$ 的畸变。两种拍法各标定一次：标定板只出现在画面中央，或者覆盖全画面，各 12 张。然后另拍 30 张覆盖全画面的图，**不参与标定**，用标定出的内参去算它们的重投影误差，并且把画面中央和四角分开统计。

先用你的线性解做（模型里没有畸变），有 OpenCV 时再用带 $k_1$、$k_2$ 的模型做一遍。

预测：线性解的两种拍法里，哪一种自己报告的 RMS 更小？它在留出视图上还是这么小吗？换成带畸变的模型之后，两种拍法自己报告的 RMS 差多少，留出视图四角的残差又差多少？

In [ ]:
held = views_k1(30, 0.2, 100, spread=0.85)
TRAIN = {"只拍画面中央": views_k1(12, 0.2, 7, spread=0.15), "覆盖全画面": views_k1(12, 0.2, 8, spread=0.85)}

def in_middle(pos):
    return (np.abs(pos[:, 0] - 320) < 110) & (np.abs(pos[:, 1] - 240) < 80)

def in_corners(pos):
    return (np.abs(pos[:, 0] - 320) > 210) & (np.abs(pos[:, 1] - 240) > 150)

def report(name, self_rms, pos, res):
    e = np.hypot(res[:, 0], res[:, 1])
    print(f"  {name}：标定自己报告的 RMS {self_rms:.2f} px；留出视图 RMS {rms_of(res):.2f} px，"
          f"其中画面中央平均 {e[in_middle(pos)].mean():.2f} px，四角平均 {e[in_corners(pos)].mean():.2f} px")
    return e

def holdout_linear():
    out = {}
    for name, train in TRAIN.items():
        K = try_calibrate(train)
        if K is None:
            print(f"  {name}：线性解解不出有效的内参")
            continue
        out[name] = (rms_of(residuals(K, train)[1]), *residuals(K, held))
    return out

result = run_or_hint(holdout_linear)
if result:
    print("线性解（模型里没有畸变）：")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for a, (name, (self_rms, pos, held_res)) in zip(ax, result.items()):
        e = report(name, self_rms, pos, held_res)
        sc = a.scatter(pos[:, 0], pos[:, 1], c=e, s=5, cmap="magma", vmin=0, vmax=3)
        a.set_xlim(0, 640); a.set_ylim(480, 0); a.set_aspect("equal"); a.set_title(f"留出残差（标定时{name}）")
    fig.colorbar(sc, ax=ax, label="px"); plt.show()

if cv2 is not None:
    print("OpenCV，模型里有 k1、k2：")
    obj = BOARD3.astype(np.float32)
    for name, train in TRAIN.items():
        Kcv, dist, rms, _, _ = cv_calibrate(train, True)
        pos, res = [], []
        for uv, _, _ in held:
            _, rv, tv = cv2.solvePnP(obj, uv.astype(np.float32).reshape(-1, 1, 2), Kcv, dist)
            pos.append(uv); res.append(uv - cv2.projectPoints(obj, rv, tv, Kcv, dist)[0].reshape(-1, 2))
        report(name, rms, np.vstack(pos), np.vstack(res))
        print(f"    估出的 k1 = {dist.ravel()[0]:+.3f}（真值 -0.100），fx = {Kcv[0, 0]:.1f}（真值 820）")
if result or cv2 is not None:
    print("想一想：哪些数字是标定程序自己会报告的，哪些只有留出验证并且按位置看才看得到？这里每种拍法只标定了一次，教程图 5 是重复 30 次的统计。（教程第 6.3 节）")

## 破坏实验与记录

- 把某张视图的角点顺序打乱，结果怎样？
- 把棋盘格边长写错（15 写成 10），哪些参数受影响，哪些不受影响？
- 实验 5 里把 `views_k1(40, 0.2, 3)` 的 `k1` 换成 +0.1（枕形畸变），残差曲线怎么变？再换几个随机种子，fx 的偏差方向固定吗？
- 把实验 1 的曲线、实验 2 和实验 4 的结论、实验 6 的那张表写进 `实验记录.md`。